In [ ]:
import pandas as pd

pd.options.display.max_columns = None


UK_AUDIO_PATH = (
    "<path to the COVID-19 Sounds release>/covid19_data_0426"
)

android = pd.read_csv(
    "all_metadata/results_raw_20210426_lan_yamnet_android_noloc.csv",
    sep=";",
    index_col=0,
)
ios = pd.read_csv(
    "all_metadata/results_raw_20210426_lan_yamnet_ios_noloc.csv", sep=";", index_col=0
)
# web = pd.read_csv(
#     "all_metadata/results_raw_20210426_lan_yamnet_web_noloc.csv", sep=";", index_col=0
# )
all_data = pd.concat([android, ios])

medhistory_dummies = all_data["Medhistory"].str.get_dummies(sep=",")
all_data = pd.concat([all_data, medhistory_dummies], axis=1)

symptom_dummies = all_data["Symptoms"].str.get_dummies(sep=",")
all_data = pd.concat([all_data, symptom_dummies], axis=1)
all_data.head()

In [ ]:
all_data["Voice filename"].str.split("_").str[0]

In [ ]:
all_data["audio_sample_path"] = (
    UK_AUDIO_PATH
    + "/"
    + all_data["Uid"]
    + "/"
    + all_data["Folder Name"]
    + "/"
    + (all_data["Voice filename"].str.split(".").str[0] + ".wav")
)
all_data.head()

In [ ]:
all_data["date"] = (
    all_data.Date.str[:10]
    .str.split("-")
    .apply(lambda x: "-".join(x[::-1]) if len(x[-1]) == 4 else "-".join(x))
)

In [ ]:
all_data[((all_data.Smoking != "never") & (all_data.copd == 0))]

In [ ]:
# Filter

all_data = all_data[(
    (all_data.Hospitalized == "no")  # out: hospitalized individuals
    & (all_data["Voice check"] == "v")  # in: recording contains voice
    & (all_data["Covid-Tested"].str.lower().str.contains("never|negative|no"))  # out: Covid positive
    & (all_data["pnts"] == 0).all(axis=1)  # out: might have conditions or symptoms but prefers not to say
    & (~((all_data.Smoking != "never") & (all_data.copd == 0)))  # out: non-COPD cases with smoking history
    & (all_data.lung == 0)  # out: lung disease
    & (all_data.pulmonary == 0)  # out: pulmonary disease
)]

In [ ]:
import numpy as np

def process_age(x):
    # Handle NaN, None, or empty values
    if pd.isna(x) or x == "" or x is None:
        return np.nan

    # Ensure x is a string
    x = str(x).strip()

    # Handle empty string after stripping
    if x == "":
        return np.nan

    # Handle age ranges like "25-30"
    if "-" in x:
        try:
            ages = [int(i.strip()) for i in x.split("-") if i.strip().isdigit()]
            if ages:  # Only if we have valid ages
                return int(np.median(ages))
            else:
                return np.nan
        except (ValueError, TypeError):
            return np.nan

    # Handle single age values like "25"
    elif x.isdigit():
        try:
            return int(x)
        except (ValueError, TypeError):
            return np.nan

    # Handle any other case
    else:
        return np.nan


all_data["age"] = all_data["Age"].apply(process_age)
all_data.loc[pd.isna(all_data.age), "age"] = (
    all_data.age.median()
)
all_data["age"] = all_data["age"].astype(int)

all_data["sex"] = all_data.Sex.map(
    {
        "Female": "w",
        "Male": "m",
    }
)

unique_uids = all_data["Uid"].unique()
uid_to_id = {uid: i for i, uid in enumerate(unique_uids, start=3001)}
all_data["audio_id"] = all_data["Uid"].map(uid_to_id)

In [ ]:
all_data.columns = all_data.columns.str.strip()

all_data.drop(
    columns=[
        "Age",
        "Sex",
        "Smoking",
        "Date",
        "Medhistory",
        "Folder Name",
        "Symptoms",
        "Voice check",
        "Cough check",
        "Breath check",
        "Sampling Rate",
        "None",
        "pnts",
        "Symptoms",
        "Covid-Tested",
        "Hospitalized",
        "Voice filename",
        "Cough filename",
        "Breath filename",
        "None"
    ],
    errors="ignore",
    inplace=True
)

In [ ]:
all_data.rename(
    columns={
        "Uid": "uid",
        "Medhistory": "med_history",
        "Smoking": "smoking",
        "Language": "language",
        "Symptoms": "symptoms",
        "Covid-Tested": "covid_tested",
        "Sampling Rate": "sampling_rate",
    },
    inplace=True,
    errors="ignore",
)

In [ ]:
all_data.head()

In [ ]:
all_data.loc[all_data.copd == 1].uid.nunique(), all_data.loc[all_data.copd == 0].uid.nunique()

In [ ]:
all_data.to_csv("all_metadata/uk_metadata_cleaned.csv", index=False)

In [ ]:
all_data.drop_duplicates(subset=["uid"], keep="first").to_csv("all_metadata/uk_metadata_cleaned_deduplicated.csv", index=False)

## Filter out symptoms and other diseases

In [ ]:
no_relevant_conditions = all_data[
    (all_data.angina == 0)
    & (all_data.asthma == 0)
    & (all_data.cancer == 0)
    & (all_data.cystic == 0)
    & (all_data.heart == 0)
    & (all_data.hiv == 0)
    & (all_data.long == 0)
    & (all_data.longterm == 0)
    & (all_data.organ == 0)
    & (all_data.otherHeart == 0)
    & (all_data.stroke == 0)
    & (all_data.valvular == 0)
    & (all_data.chills == 0)
    & (all_data.dizziness == 0)
    & (all_data.drycough == 0)
    & (all_data.fever == 0)
    & (all_data.headache == 0)
    & (all_data.muscleache == 0)
    & (all_data.never == 0)
    & (all_data.runny == 0)
    & (all_data.runnyblockednose == 0)
    & (all_data.shortbreath == 0)
    & (all_data.smelltasteloss == 0)
    & (all_data.sorethroat == 0)
    & (all_data.tightness == 0)
    & (all_data.wetcough == 0)
].copy()

copd = no_relevant_conditions[(no_relevant_conditions.copd == 1)]
control = no_relevant_conditions[(no_relevant_conditions.copd == 0)]

copd.uid.nunique(), control.uid.nunique()

In [ ]:
no_relevant_conditions.loc[no_relevant_conditions.copd == 1, "lung_disease_main"] = (
    "copd"
)
no_relevant_conditions.loc[no_relevant_conditions.copd == 0, "lung_disease_main"] = (
    "control"
)
no_relevant_conditions.loc[no_relevant_conditions.copd == 1, "control"] = 0
no_relevant_conditions.loc[no_relevant_conditions.copd == 0, "control"] = 1

no_relevant_conditions["longitudinal"] = 0
no_relevant_conditions["recording_category"] = "poem"

In [ ]:
no_relevant_conditions.loc[no_relevant_conditions.audio_id == 3146]

In [ ]:
# Drop duplicates, keep most recent recording per category
no_relevant_conditions.sort_values(by=["audio_id", "date"], ascending=False, inplace=True)
no_relevant_conditions.drop_duplicates("audio_id", inplace=True)

In [ ]:
# TODO check that we drop non-COPD cases that smoke

no_relevant_conditions.to_csv("all_metadata/uk_metadata_no_other_condition.csv", index=False)

In [ ]:
no_relevant_conditions.loc[no_relevant_conditions.language == "de"].lung_disease_main.value_counts()

In [ ]:
unique_uids = no_relevant_conditions["uid"].unique()

uid_to_id = {uid: i for i, uid in enumerate(unique_uids, start=3001)}

no_relevant_conditions["audio_id"] = no_relevant_conditions["uid"].map(uid_to_id)

no_relevant_conditions

In [ ]:
no_relevant_conditions.uid.value_counts(dropna=False)

In [ ]:
copd.drop_duplicates("uid").language.value_counts()

In [ ]:
control.drop_duplicates("uid").language.value_counts(dropna=False)

In [ ]:
all_data.Symptoms.str.split(",").explode().value_counts(dropna=False)

In [ ]:
all_data.Smoking.value_counts(dropna=False)

In [ ]:
all_data["Covid-Tested"].value_counts(dropna=False)

In [ ]:
import pandas as pd

md_split = pd.read_csv(
    "metadata_post_splitting.csv"
)
md_split.head()

In [ ]:
md_split[md_split.duplicated("recording_identifier")].sort_values(
    "recording_identifier"
)

In [ ]:
md_split.chunk_number.value_counts(dropna=False)

In [ ]:
md_split[md_split.duplicated("recording_identifier")].loc[
    pd.isna(md_split.chunk_number)
]

In [ ]:
sorted(list(md_split.columns))

In [ ]:
pd.options.display.max_columns = None
md_split.loc[md_split.audio_id == 234].sort_values(["date", "chunk_number"])[
    [
        "audio_id",
        "date",
        "recording_category",
        "recording_identifier",
        "chunk_number",
        "audio_sample_path",
    ]
]

In [ ]:
merge_df = pd.read_csv("merged_metadata.csv")
merge_df.head()

In [ ]:
merge_df[merge_df.duplicated("recording_identifier")]

In [ ]:
merge_df[pd.isna(merge_df.recording_identifier)]